In [ ]:
import torch
import numpy as np
from PIL import Image
from transformers import AutoModelForVision2Seq
from transformers import AutoProcessor
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(
    "openvla/openvla-7b",
    trust_remote_code=True,
)
print("Processor type:", type(processor).__name__)

vla = AutoModelForVision2Seq.from_pretrained(
    "openvla/openvla-7b",
    attn_implementation="eager",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    quantization_config=bnb_config,
)

print("Model 로드 완료")
print(f"GPU 메모리 사용: {torch.cuda.memory_allocated()/1e9:.2f} GB")

image = Image.fromarray(
    (np.random.rand(224, 224, 3) * 255).astype(np.uint8)
)
instruction = "pick up the can"
prompt = f"In: What action should the robot take to {instruction}?\nOut:"

inputs = processor(prompt, image).to("cuda:0", dtype=torch.float16)
print(f"inputs.keys(): {list(inputs.keys())}")

# attention_mask 는 전달하지 않는다 -- predict_action 이 빈 토큰(29871) 을 input_ids 에만
# 덧붙여 mask 와 길이가 1 어긋나므로 (eager attention 에서 크래시), generate 가 mask 를
# 알아서 생성하게 둔다
with torch.no_grad():
    action = vla.predict_action(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        unnorm_key="bridge_orig",
        do_sample=False,
    )
print(f"Action shape: {action.shape}")
print(f"Action : {action}")
print(f"GPU 메모리 : {torch.cuda.memory_allocated()/1e9:.2f} GB")
